# 01 — Ground Tabular Data: Initial EDA and Schema Standardization


## Purpose
This notebook performs a first-pass exploratory analysis of the ground (tabular) datasets for both sites.
It focuses on:
- Loading and validating raw files
- Identifying timestamp and target (GHI) columns
- Verifying time frequency and missingness patterns
- Producing a standardized, site-agnostic view of the data schema

## Imports and settings

In [1]:
from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data_raw"

UNI_PATH = DATA_RAW / "Data_Uniandes_tierra.csv"  
ELP_PATH = DATA_RAW / "Data_ElPaso_tierra.csv"

TIMEZONE = "America/Bogota"
FREQ = "10min"   # expected sampling frequency for this project

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UNI_PATH exists:", UNI_PATH.exists(), "->", UNI_PATH)
print("ELP_PATH exists:", ELP_PATH.exists(), "->", ELP_PATH)


PROJECT_ROOT: /srv/projects/Proyecto_e_ladino
UNI_PATH exists: True -> /srv/projects/Proyecto_e_ladino/data_raw/Data_Uniandes_tierra.csv
ELP_PATH exists: True -> /srv/projects/Proyecto_e_ladino/data_raw/Data_ElPaso_tierra.csv


## Uniandes

In [2]:
uni_df = pd.read_csv(UNI_PATH, sep=";")

print("UNI shape:", uni_df.shape)
print("\nUNI columns:")
print(list(uni_df.columns))

print("\nUNI head:")
display(uni_df.head())


UNI shape: (165312, 7)

UNI columns:
['Timestamp', 'Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1', 'Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1', 'Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D', 'Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD', 'Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T', 'Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S']

UNI head:


,Timestamp,Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1,Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1,Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D,Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD,Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T,Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S
0,2023-09-01 00:00:00,"73,896","745,288","163,000","0,000","12,718","1,042"
1,2023-09-01 00:05:00,"74,888","745,184","108,532","0,000","12,292","2,858"
2,2023-09-01 00:10:00,"76,484","745,140","113,694","0,000","11,932","2,728"
3,2023-09-01 00:15:00,"76,674","745,032","110,576","0,144","12,132","2,300"
4,2023-09-01 00:20:00,"75,824","744,972","116,006","0,000","12,318","1,346"


## El Paso

In [3]:
elp_df = pd.read_csv(ELP_PATH)

print("ELP shape:", elp_df.shape)
print("\nELP columns:")
print(list(elp_df.columns))

print("\nELP head:")
display(elp_df.head())


ELP shape: (107172, 10)

ELP columns:
['Unnamed: 0', 'CSI', 'GHI', 'Presion', 'TempAmb', 'Wind Y', 'Wind X', 'DoY Sin', 'DoY Cos', 'horas']

ELP head:


,Unnamed: 0,CSI,GHI,Presion,TempAmb,Wind Y,Wind X,DoY Sin,DoY Cos,horas
0,2022-02-21 18:00:00+00:00,2.0,3.0352,1000.7912,29.9672,2.832954,-0.093612,0.778764,0.627317,18
1,2022-02-21 18:10:00+00:00,0.0,0.3562,1000.9321,29.5689,3.387552,0.796801,0.778764,0.627317,18
2,2022-02-21 18:20:00+00:00,0.0,0.0000,1001.1479,29.2593,2.091197,-0.878680,0.778764,0.627317,18
3,2022-02-21 18:30:00+00:00,0.0,0.0000,1001.2992,28.9183,-0.487957,-1.478562,0.778764,0.627317,18
4,2022-02-21 18:40:00+00:00,0.0,0.0000,1001.4676,28.5578,0.891171,-2.047462,0.778764,0.627317,18


## Sanity check

In [4]:
print("UNI dtypes:")
display(uni_df.dtypes)

print("\nELP dtypes:")
display(elp_df.dtypes)

UNI dtypes:


Timestamp                                                                                              str
Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1      str
Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1    str
Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D                 str
Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD                          str
Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T                             str
Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S               str
dtype: object


ELP dtypes:


Unnamed: 0        str
CSI           float64
GHI           float64
Presion       float64
TempAmb       float64
Wind Y        float64
Wind X        float64
DoY Sin       float64
DoY Cos       float64
horas           int64
dtype: object

## Standardize columns

In [5]:
uni_rename_map = {
    # timestamp
    "Timestamp": "timestamp",

    # radiation
    "Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD": "ghi",

    # meteorology
    "Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T": "air_temperature_c",
    "Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1": "relative_humidity_pct",
    "Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1": "air_pressure_hpa",
    "Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S": "wind_speed_ms",
    "Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D": "wind_direction_deg",
}

uni_df = uni_df.rename(columns=uni_rename_map)

print("UNI standardized columns:")
print(list(uni_df.columns))
display(uni_df.head())


UNI standardized columns:
['timestamp', 'relative_humidity_pct', 'air_pressure_hpa', 'wind_direction_deg', 'ghi', 'air_temperature_c', 'wind_speed_ms']


,timestamp,relative_humidity_pct,air_pressure_hpa,wind_direction_deg,ghi,air_temperature_c,wind_speed_ms
0,2023-09-01 00:00:00,"73,896","745,288","163,000","0,000","12,718","1,042"
1,2023-09-01 00:05:00,"74,888","745,184","108,532","0,000","12,292","2,858"
2,2023-09-01 00:10:00,"76,484","745,140","113,694","0,000","11,932","2,728"
3,2023-09-01 00:15:00,"76,674","745,032","110,576","0,144","12,132","2,300"
4,2023-09-01 00:20:00,"75,824","744,972","116,006","0,000","12,318","1,346"


In [6]:
elp_rename_map = {
    "Unnamed: 0": "timestamp",
    "GHI": "ghi",
    "TempAmb": "air_temperature_c",
    "Presion": "air_pressure_hpa",
    "CSI": "clear_sky_index",
    "Wind X": "wind_x",
    "Wind Y": "wind_y",
    "horas": "hour_of_day",
    "DoY Sin": "doy_sin",
    "DoY Cos": "doy_cos",
}

elp_df = elp_df.rename(columns=elp_rename_map)

print("ELP standardized columns:")
print(list(elp_df.columns))
display(elp_df.head())


ELP standardized columns:
['timestamp', 'clear_sky_index', 'ghi', 'air_pressure_hpa', 'air_temperature_c', 'wind_y', 'wind_x', 'doy_sin', 'doy_cos', 'hour_of_day']


,timestamp,clear_sky_index,ghi,air_pressure_hpa,air_temperature_c,wind_y,wind_x,doy_sin,doy_cos,hour_of_day
0,2022-02-21 18:00:00+00:00,2.0,3.0352,1000.7912,29.9672,2.832954,-0.093612,0.778764,0.627317,18
1,2022-02-21 18:10:00+00:00,0.0,0.3562,1000.9321,29.5689,3.387552,0.796801,0.778764,0.627317,18
2,2022-02-21 18:20:00+00:00,0.0,0.0000,1001.1479,29.2593,2.091197,-0.878680,0.778764,0.627317,18
3,2022-02-21 18:30:00+00:00,0.0,0.0000,1001.2992,28.9183,-0.487957,-1.478562,0.778764,0.627317,18
4,2022-02-21 18:40:00+00:00,0.0,0.0000,1001.4676,28.5578,0.891171,-2.047462,0.778764,0.627317,18


### Timestamps & data ranges

In [7]:
# Parse timestamps
uni_df["timestamp"] = pd.to_datetime(uni_df["timestamp"], errors="coerce")
elp_df["timestamp"] = pd.to_datetime(elp_df["timestamp"], errors="coerce")

# Drop rows with invalid timestamps
uni_df = uni_df.dropna(subset=["timestamp"])
elp_df = elp_df.dropna(subset=["timestamp"])

# Set index
uni_df = uni_df.set_index("timestamp").sort_index()
elp_df = elp_df.set_index("timestamp").sort_index()

print("UNI date range:", uni_df.index.min(), "→", uni_df.index.max())
print("ELP date range:", elp_df.index.min(), "→", elp_df.index.max())

print("\nUNI rows:", len(uni_df))
print("ELP rows:", len(elp_df))

UNI date range: 2023-09-01 00:00:00 → 2025-03-27 23:55:00
ELP date range: 2022-02-21 18:00:00+00:00 → 2024-03-06 23:50:00+00:00

UNI rows: 165312
ELP rows: 107172


## Nulls/NaNs

In [8]:
def null_report(df, label):
    print(f"\n{label} — missing values (fraction):")
    rep = df.isna().mean().sort_values(ascending=False)
    print(rep[rep > 0])

null_report(uni_df, "UNI")
null_report(elp_df, "ELP")



UNI — missing values (fraction):
relative_humidity_pct    0.000998
air_pressure_hpa         0.000998
wind_direction_deg       0.000998
ghi                      0.000998
air_temperature_c        0.000998
wind_speed_ms            0.000998
dtype: float64

ELP — missing values (fraction):
Series([], dtype: float64)


## Timezone normalization (to UTC) & Time alingment

In [9]:
UNI_LOCAL_TZ = "America/Bogota"

In [10]:
if uni_df.index.tz is None:
    uni_df.index = uni_df.index.tz_localize(UNI_LOCAL_TZ)
uni_df = uni_df.tz_convert("UTC")

if elp_df.index.tz is None:
    elp_df.index = elp_df.index.tz_localize("UTC")
else:
    elp_df = elp_df.tz_convert("UTC")

print("UNI tz:", uni_df.index.tz, "| range:", uni_df.index.min(), "→", uni_df.index.max())
print("ELP tz:", elp_df.index.tz, "| range:", elp_df.index.min(), "→", elp_df.index.max())

UNI tz: UTC | range: 2023-09-01 05:00:00+00:00 → 2025-03-28 04:55:00+00:00
ELP tz: UTC | range: 2022-02-21 18:00:00+00:00 → 2024-03-06 23:50:00+00:00


In [11]:
FREQ = "10min"

def reindex_to_grid(df: pd.DataFrame, freq: str) -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    start = df.index.min().floor(freq)
    end = df.index.max().ceil(freq)
    grid = pd.date_range(start, end, freq=freq, tz=df.index.tz)
    df_grid = df.reindex(grid)
    return df_grid, grid

uni_grid, uni_idx = reindex_to_grid(uni_df, FREQ)
elp_grid, elp_idx = reindex_to_grid(elp_df, FREQ)

print("UNI original rows:", len(uni_df), "| on-grid rows:", len(uni_grid))
print("ELP original rows:", len(elp_df), "| on-grid rows:", len(elp_grid))

UNI original rows: 165312 | on-grid rows: 82657
ELP original rows: 107172 | on-grid rows: 107172


### Coverage stats

In [12]:
def coverage_stats(df_grid: pd.DataFrame, key_col: str) -> dict:
    out = {}
    out["rows_total"] = int(len(df_grid))
    out["rows_key_nonnull"] = int(df_grid[key_col].notna().sum())
    out["coverage_key"] = float(df_grid[key_col].notna().mean())

    miss = df_grid[key_col].isna().astype(int)
    
    runs = (miss.diff().fillna(0) != 0).cumsum()
    gap_lengths = miss.groupby(runs).sum()
 
    gap_lengths = gap_lengths[gap_lengths > 0]

    out["n_gaps"] = int(len(gap_lengths))
    if len(gap_lengths) > 0:
        out["gap_steps_median"] = float(np.median(gap_lengths))
        out["gap_steps_p95"] = float(np.percentile(gap_lengths, 95))
        out["gap_minutes_p95"] = float(np.percentile(gap_lengths, 95) * 10.0)
    else:
        out["gap_steps_median"] = 0.0
        out["gap_steps_p95"] = 0.0
        out["gap_minutes_p95"] = 0.0

    return out

uni_cov = coverage_stats(uni_grid, "ghi")
elp_cov = coverage_stats(elp_grid, "ghi")

print("UNI coverage stats:", uni_cov)
print("ELP coverage stats:", elp_cov)


UNI coverage stats: {'rows_total': 82657, 'rows_key_nonnull': 82573, 'coverage_key': 0.9989837521323058, 'n_gaps': 6, 'gap_steps_median': 1.0, 'gap_steps_p95': 59.5, 'gap_minutes_p95': 595.0}
ELP coverage stats: {'rows_total': 107172, 'rows_key_nonnull': 107172, 'coverage_key': 1.0, 'n_gaps': 0, 'gap_steps_median': 0.0, 'gap_steps_p95': 0.0, 'gap_minutes_p95': 0.0}


## Export

In [13]:
OUT_DIR = PROJECT_ROOT / "data" / "ground_aligned"
OUT_DIR.mkdir(parents=True, exist_ok=True)

uni_out = OUT_DIR / "ground_10min_utc_uniandes.parquet"
elp_out = OUT_DIR / "ground_10min_utc_elpaso.parquet"

uni_grid.to_parquet(uni_out)
elp_grid.to_parquet(elp_out)

print("Exported:")
print(" -", uni_out)
print(" -", elp_out)

Exported:
 - /srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_uniandes.parquet
 - /srv/projects/Proyecto_e_ladino/data/ground_aligned/ground_10min_utc_elpaso.parquet
